# Week 8 Lab Exercise — ARIA v5.0: The Matai'an Three-Act Auditor
# 第八週課堂練習 — ARIA v5.0：馬太鞍三幕稽核器

**Guided lab worksheet** — follow along with the instructor's Demo notebook and fill in the TODO cells.
**引導式練習** — 跟著老師的 Demo 操作，完成 TODO 標記的程式碼。

**Case / 案例**: 2025 Matai'an Creek Barrier Lake Event（馬太鞍溪堰塞湖事件）

---

### Class rhythm / 課程節奏

| Phase / 階段 | Content / 內容 | Cells | Time / 時間 |
|---|---|---|---|
| **Lab 1** | Spectral Detective 光譜偵探 | [S1]–[S6] | 35 min |
| **Lab 2** | Three-Act Audit 三幕稽核 | [S7]–[S15] | 50 min |


## Lab 1 — Environment + Three-Act STAC Scene Selection
## Lab 1 — 環境設定 + 三幕 STAC 影像選擇


In [1]:
# [S1] Environment setup — fill in the imports and STAC client
# TODO: import pystac_client, planetary_computer as pc, stackstac, rioxarray as rxr
# TODO: import xarray as xr, numpy as np, pandas as pd, matplotlib.pyplot as plt
# TODO: import geopandas as gpd; from shapely.geometry import Point

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Microsoft JhengHei for Chinese labels
plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "PingFang TC", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

# TODO: Connect to Planetary Computer
# catalog = pystac_client.Client.open(
#     "https://planetarycomputer.microsoft.com/api/stac/v1",
#     modifier=pc.sign_inplace,
# )

# --- IMPORTANT: MPC-friendly search pattern ---
# MPC's STAC API sometimes returns 504 "maximum allowed time" errors when you
# combine bbox + date + server-side query={"eo:cloud_cover": {"lt": N}}.
# Safer pattern: skip the server-side query, cap max_items, and filter
# cloud_cover CLIENT-SIDE. Wrap the whole thing in a retry loop.
#
# Reference implementation (uncomment and adapt):
#
# def robust_search(bbox, datetime_range, cloud_max=30, max_items=60, tries=3):
#     for attempt in range(tries):
#         try:
#             search = catalog.search(
#                 collections=["sentinel-2-l2a"],
#                 bbox=bbox,
#                 datetime=datetime_range,
#                 max_items=max_items,
#             )
#             items = list(search.items())
#             items = [i for i in items if i.properties.get("eo:cloud_cover", 100) < cloud_max]
#             items.sort(key=lambda i: i.properties["eo:cloud_cover"])
#             return items
#         except Exception as e:
#             time.sleep(2 ** attempt)
#     raise RuntimeError("STAC search failed 3x")

# AOI — Matai'an catchment (upper Wanrong → downstream Guangfu)
MATAIAN_BBOX = [121.28, 23.56, 121.52, 23.76]
WANTED_BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]

# --- Fill these in after Step A.3 (reproducible scene IDs) ---
PRE_ITEM_ID  = "FILL_IN_AFTER_QA"
MID_ITEM_ID  = "FILL_IN_AFTER_QA"
POST_ITEM_ID = "FILL_IN_AFTER_QA"


In [2]:
# [S2] ACT 1 — Pre-event STAC search (Jun 2025, before Typhoon Wipha)
# TODO: use robust_search() from S1 — MATAIAN_BBOX, "2025-06-01/2025-07-15", cloud_max=20
# items_pre = robust_search(MATAIAN_BBOX, "2025-06-01/2025-07-15", cloud_max=20)

# TODO: TCI Quick-QA the top 3 candidates in a dynamic 1xN subplot grid
#       n = max(min(len(items_pre), 3), 1)
#       fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
#       if n == 1: axes = [axes]
# Use item.assets["visual"].href and rxr.open_rasterio(..., overview_level=3)

# TODO: Pick your best scene and save the item ID
# pre_item = items_pre[0]
# PRE_ITEM_ID = pre_item.id


### ✏️ Discussion / 討論（Act 1 — Pre）
Why did you pick this Pre-event scene? Consider cloud cover **over the Matai'an valley** specifically.
為什麼選這張事件前影像？特別注意馬太鞍谷地上方的雲量（不只是 tile 整體數字）。

*(Write 1-2 sentences here / 寫 1-2 句)*


In [ ]:
# [S3] ACT 2 — Mid-event STAC search (Aug–Sep 2025, barrier lake present, before Sep 23 breach)
# TODO: robust_search for 2025-08-01/2025-09-20, cloud_max=40 (monsoon — relax clouds)
# items_mid = robust_search(MATAIAN_BBOX, "2025-08-01/2025-09-20", cloud_max=40)

# TODO: TCI Quick-QA top 3 — LOOK FOR the barrier lake! It should be visible as a new
#       turquoise patch in the upper Matai'an valley.

# TODO: Pick best, save item ID
# mid_item = items_mid[0]
# MID_ITEM_ID = mid_item.id


### ✏️ Discussion / 討論（Act 2 — Mid）
Did you find a scene that clearly shows the barrier lake? Why did you pick this one?
你找到了清楚顯示堰塞湖的影像嗎？為什麼選這一張？

*(Write 1-2 sentences here / 寫 1-2 句)*


In [ ]:
# [S4] ACT 3 — Post-event STAC search (after Sep 23 breach)
# TODO: robust_search for 2025-09-25/2025-11-15, cloud_max=30
# items_post = robust_search(MATAIAN_BBOX, "2025-09-25/2025-11-15", cloud_max=30)

# TODO: TCI Quick-QA top 3 — look for evidence of the drained lake bed + fresh sediment around Guangfu
# TODO: pick best, save item ID
# post_item = items_post[0]
# POST_ITEM_ID = post_item.id


### ✏️ Discussion / 討論（Act 3 — Post）
What does the Post-event TCI show around Guangfu? Can you see evidence of the debris flow?
事件後的 TCI 在光復周邊顯示了什麼？你看得到土石流的證據嗎？

*(Write 1-2 sentences here / 寫 1-2 句)*


## Stream the Three Cubes / 串流三個立方體


In [ ]:
# [S5] Stream function — returns a rescaled xarray cube for any item
# HINT: Sentinel-2 URLs on Planetary Computer carry SAS tokens that
# expire in ~1 hour. Call pc.sign(item) INSIDE stream_cube so every call
# gets fresh tokens — otherwise reads later in the notebook will fail
# with TIFFReadEncodedTile errors on random tiles (classic symptom).
def stream_cube(item, bands=WANTED_BANDS, bbox=MATAIAN_BBOX):
    # TODO: signed = pc.sign(item)   # refresh SAS tokens right before reading
    # TODO: use stackstac.stack([signed], assets=bands, epsg=32651,
    #       resolution=10, bounds_latlon=bbox, chunksize=2048)
    # TODO: .squeeze("time") and divide by 10000.0 to get surface reflectance
    pass

# ── Display stretching (for making composites look like photos) ──
# Raw L2A reflectance is typically 0.02–0.50, monitors need 0–1 RGB.
# You need a stretch function to bridge the gap. TWO approaches:
#
#   (1) Uniform ×scale stretch — fast, primitive. One scale factor
#       applied to all three channels. B08 tends to saturate (neon green).
#
#   (2) Per-channel percentile stretch — industry standard. Each band
#       independently rescaled so its [2%, 98%] maps to [0, 1]. No
#       saturation, balanced contrast, natural colors.
#
# NOTE: stretching is ONLY for display. Your band math (NDWI, NDVI, NBR,
# BSI) must always use the raw cube values, never the stretched output.

def composite_stretched(cube, r, g, b, lo=2, hi=98):
    # TODO: rgb = np.stack([cube.sel(band=r), cube.sel(band=g), cube.sel(band=b)], axis=-1)
    # TODO: for each channel k: compute np.nanpercentile(rgb[...,k], [lo, hi])
    # TODO: out[...,k] = np.clip((rgb[...,k] - p_lo) / (p_hi - p_lo), 0, 1)
    # TODO: return out (float32)
    pass

# cube_pre  = stream_cube(pre_item)
# cube_mid  = stream_cube(mid_item)
# cube_post = stream_cube(post_item)


## Lab 1 (cont.) — Four Change Metric Functions / 四個變遷指標函式

Implement NDWI, NDVI, BSI, NBR — same formulas as the Demo notebook.
實作 NDWI、NDVI、BSI、NBR — 公式和 Demo notebook 相同。


In [ ]:
# [S6] Four reusable change metric functions
def nir_drop(pre, post):
    # TODO: return pre.sel(band="B08") - post.sel(band="B08")
    pass

def swir_post(post):
    # TODO: return post.sel(band="B12")
    pass

def bsi(cube):
    # TODO: BSI = ((B11 + B04) - (B08 + B02)) / ((B11 + B04) + (B08 + B02))
    pass

def bsi_change(pre, post):
    # TODO: return bsi(post) - bsi(pre)
    pass

def ndvi(cube):
    # TODO: (B08 - B04) / (B08 + B04)
    pass

def ndvi_change(pre, post):
    # TODO: return ndvi(pre) - ndvi(post)
    pass


## Lab 2 — Three Detection Masks / 三張偵測遮罩

### C1. Barrier lake mask / 堰塞湖遮罩（Pre → Mid）

Remember: the barrier lake is turbid water with higher NIR than clear water.
提醒：堰塞湖是高濁度水體，NIR 比清水高。


In [ ]:
# [S7] Barrier lake detection
# Baseline rule: (nir_pre > 0.25) & (nir_mid < 0.18) & (blue_mid > 0.03) & (green_mid > nir_mid)
# IMPORTANT: The barrier lake is TURBID (sediment-laden), NIR ~0.10-0.18
#            Using a clear-water threshold like 0.08 will return ZERO lake pixels!

# TODO: compute nir_pre, nir_mid, blue_mid, green_mid from cube_pre and cube_mid
# TODO: try 3 candidate values for the nir_mid upper bound: 0.12, 0.15, 0.18
# TODO: also add green_mid > nir_mid gate (water-like spectral shape)
# TODO: add upstream spatial gate: restrict to west of 121.33°E to exclude
#       downstream river flooding false positives (see Demo Cell [16])
# TODO: report the resulting lake area in km² for each threshold
# TODO: pick the value that best matches the NCDR reference ~0.86 km² (Sep 11 peak) and explain why

# lake_mask = ...
# print(f"Lake area: {float(lake_mask.sum().values) * 100 / 1e6:.3f} km²")

# 📊 TODO (REQUIRED — save to output/07_lake_mask.png):
#   Plot a 1x2 panel:
#     ax[0] = Mid-event TCI (composite B04/B03/B02 from cube_mid)
#     ax[1] = same TCI + lake_mask overlayed in cyan (use np.where(lake_mask, 1, np.nan))
#   Title: "ACT 1→2 Barrier lake detection — NCDR ref ≈0.86 km²"
#   Expected: you should see a contiguous blue patch in the upper Matai'an valley
#             centered around (121.292, 23.696).
#   If the overlay is scattered/noisy, check your spatial gate and threshold.


### C2. Landslide source scar mask / 崩塌源區遮罩（Pre → Post）


In [ ]:
# [S8] Ground truth collection — 10+10 points
# Landslide truth points (lon, lat) — fill in from reference gpkg, georeferenced news photos, or QGIS
landslide_truth = [
    # (lon, lat, "Y"),   # 10 confirmed landslide pixels in the upper Matai'an headwall
    # ...
]
stable_truth = [
    # (lon, lat, "N"),   # 10 stable forest pixels nearby
    # ...
]

# TODO: convert both lists to GeoDataFrames in EPSG:4326, then reproject to cube_post.rio.crs
# TODO: sample the masks at each truth point


In [ ]:
# [S8b] Threshold tuning — 5 candidate pairs
candidate_thresholds = [
    # (nir_drop_min, swir_post_min)
    (0.10, 0.20),
    (0.15, 0.25),   # baseline
    (0.20, 0.30),
    (0.15, 0.30),
    (0.20, 0.25),
]

# TODO: For each pair, build the landslide mask, sample at the 20 truth points,
#       compute TP/FP/TN/FN, then F1.
# TODO: Report the best F1 and its threshold pair in a markdown table.

from sklearn.metrics import f1_score  # or implement by hand

results = []
for nd_min, sw_min in candidate_thresholds:
    # mask = (nir_drop > nd_min) & (swir_post > sw_min) & (nir_pre > 0.25)
    # f1 = ...
    # results.append({"nir_drop_min": nd_min, "swir_post_min": sw_min, "F1": f1})
    pass

# pd.DataFrame(results).sort_values("F1", ascending=False)

# 📊 TODO (REQUIRED — save to output/08_landslide_threshold_grid.png):
#   Plot a 1x5 grid: each panel shows one candidate mask overlayed on the Post TCI.
#   Panel title: f"nd>{nd_min}, sw>{sw_min}  F1={f1:.2f}"
#   This is your ground-truth debugging view — the eye catches errors the F1 score misses:
#     • False positives over the main river → swir_post threshold too low
#     • Missing pixels in the headwall     → nir_drop threshold too high
#   Put a red frame around the panel with the highest F1.


### C3. Debris flow footprint mask / 土石流鋪面遮罩（Pre → Post, downstream only）


In [ ]:
# [S9] Debris flow — downstream of Matai'an Creek mouth (lon > 121.35)
# Rule: (ndvi_change > 0.25) & (bsi_change > 0.10) & (nir_pre > 0.20) & downstream_gate

# TODO: build the downstream gate using the cube's x-coordinate and pyproj
#       from pyproj import Transformer
#       tf = Transformer.from_crs("EPSG:4326", "EPSG:32651", always_xy=True)
#       _x_debris_gate, _ = tf.transform(121.35, 23.69)
#       downstream_gate = cube_post.x > _x_debris_gate
# TODO: build the debris mask, subtract lake_mask and landslide_mask
# TODO: report debris area in km²

# 📊 TODO (REQUIRED — save to output/09_debris_mask.png):
#   Plot a 1x3 panel:
#     ax[0] = ndvi_change (cmap="RdYlGn_r", vmin=-0.5, vmax=0.5) + colorbar
#     ax[1] = bsi_change  (cmap="YlOrBr",   vmin=-0.3, vmax=0.3) + colorbar
#     ax[2] = Post TCI + debris_mask overlayed in copper
#   Title: "ACT 1→3 Downstream debris flow — Guangfu sediment sheet"
#   If ax[2] shows polygons upstream of lon=121.35 something is wrong with the downstream gate.


### ✏️ Discussion / 討論：Why is the debris rule different from C2?
Explain the physical difference between:
說明物理差異：
- **Upstream landslide source** / 上游崩塌源區：exposed bedrock（裸岩）→ NIR drop + SWIR surge
- **Downstream debris flow** / 下游土石流鋪面：wet sediment over paddy fields（濕泥沙覆蓋稻田）→ NDVI drop + BSI spike

Why does each need a different band combination?
為什麼各自需要不同的波段組合？

*(Write 2-3 sentences here / 寫 2-3 句)*


In [ ]:
# [S10] Vectorize the three masks
from rasterio import features
from shapely.geometry import shape

def vectorize(mask_da, min_area):
    m = mask_da.astype("uint8").values
    polys = [shape(g) for g, v in
             features.shapes(m, mask=m.astype(bool), transform=mask_da.rio.transform())
             if v == 1]
    gdf = gpd.GeoDataFrame(geometry=polys, crs=mask_da.rio.crs)
    return gdf[gdf.area > min_area]

# TODO:
# lake_gdf       = vectorize(lake_mask,       min_area=3000)
# landslides_gdf = vectorize(landslide_mask,  min_area=2000)
# debris_gdf     = vectorize(debris_mask,     min_area=5000)

# TODO: save to mataian_detections.gpkg as three layers
# lake_gdf.to_file("mataian_detections.gpkg", layer="barrier_lake", driver="GPKG")
# landslides_gdf.to_file("mataian_detections.gpkg", layer="landslide_source", driver="GPKG")
# debris_gdf.to_file("mataian_detections.gpkg", layer="debris_flow", driver="GPKG")

# 📊 TODO (REQUIRED — save to output/10_three_masks.png):
#   Plot a 2x2 panel:
#     ax[0,0] = Mid TCI + lake_mask (cyan)              — "Barrier lake"
#     ax[0,1] = Post TCI + landslide_mask (red)         — "Landslide source"
#     ax[1,0] = Post TCI + debris_mask (copper)         — "Debris flow"
#     ax[1,1] = Post TCI + all three overlayed          — "All three overlayed"
#   This is the "Three-Act Evidence" figure — one subplot per act.
#   Good sign: the three polygons are spatially disjoint (thanks to the & ~lake_mask & ~landslide_mask gates).


## Lab 2 (cont.) — Multi-Layer Audit / 多圖層稽核


In [ ]:
# [S11] Load W3 shelters, W7 top-5 bottlenecks, and the W8 Guangfu overlay
#       (the one YOU built yourself in Pre-lab Step 7b)
# shelters = gpd.read_file("data/shelters_hualien.gpkg").to_crs(cube_post.rio.crs)
# top5     = gpd.read_file("data/top5_bottlenecks.gpkg").to_crs(cube_post.rio.crs)

# TODO: Load your own Guangfu overlay — 5 required nodes from Pre-lab Step 7b:
#       Guangfu_Station, Guangfu_Elementary, Guangfu_Township_Office,
#       Mataian_Hwy9_Bridge, Foxu_Debris_Zone  (schema: name/cn_name/node_type/priority/geometry)
# guangfu = gpd.read_file("data/guangfu_overlay.gpkg").to_crs(cube_post.rio.crs)
# assert len(guangfu) >= 5, "Pre-lab Step 7b requires 5 required nodes"
# assert guangfu.crs.to_epsg() == cube_post.rio.crs.to_epsg()


In [ ]:
# [S12] Build the Eyewitness Impact Table + Final Audit Map
# Columns:
#   asset | type | location | W4_terrain_risk | W7_centrality_rank
#        | lake_hit (Y/N) | landslide_hit (Y/N) | debris_hit (Y/N) | notes

# TODO:
#   - for each W3 shelter: check if any landslide polygon is within 100 m
#   - for each W7 bottleneck: check if any landslide polygon is within 200 m
#   - for each Guangfu node: check if within debris polygon, and within 100 m of landslide
#   - assemble the rows into a DataFrame
#   - sort: debris_hit, then landslide_hit, then W4_terrain_risk desc
#   - save to impact_table.csv

# impact_df = ...
# impact_df.to_csv("impact_table.csv", index=False)
# print(impact_df.to_markdown(index=False))

# 📊 TODO (REQUIRED — save to output/12_coverage_gap_map.png):
#   This is THE punchline figure of W8. Show, on a single 12x12 map in EPSG:32651:
#     1. Post TCI as the basemap (faded alpha=0.5)
#     2. lake_gdf       filled blue
#     3. landslides_gdf filled red
#     4. debris_gdf     filled copper
#     5. shelters       green circles, size=80, label="W3 Hualien City shelters"
#     6. top5           yellow circles, size=120, label="W7 Top-5 bottlenecks"
#     7. guangfu        red stars,    size=200, label="W8 Guangfu overlay (5 nodes)"
#   Add a text annotation: f"W3 hits: {n_w3}   W7 hits: {n_w7}   Guangfu hits: {n_guangfu} / 5"
#   Title: "Eyewitness Coverage Gap — Hualien City ARIA missed Guangfu entirely"
#   This one figure should make a 10-year-old understand why ARIA needed the Guangfu overlay.


### ✏️ Discussion / 討論：Coverage Gap Analysis / 覆蓋缺口分析

1. **W3 shelters hit? / W3 避難所被影響幾個？** How about W7 bottlenecks? / W7 瓶頸呢？
2. **Guangfu overlay nodes hit? / 光復圖層節點被影響幾個？** Compare the numbers. / 比較數字。
3. **What does this tell you** about the pre-event ARIA coverage zone? / 這對事前 ARIA 覆蓋範圍有什麼啟示？

*(Write your answers here / 在此作答)*


## Challenge / 挑戰題 — AI Advisor Operational Brief / AI 顧問應變簡報

Build a prompt and call any LLM to generate a 250-word operational brief.
組一個 Prompt，呼叫任何 LLM 產生 250 字的應變簡報。


In [ ]:
# [S13] Build the three-act AI Advisor prompt
# TODO: compute lake_km2, landslide_km2, debris_km2 from the masks
# TODO: fill in the prompt template with your impact table and detection areas

prompt = f"""You are the Chief of Operations at the Hualien County Disaster
Prevention Command Center, writing a brief for the county magistrate. ARIA v5.0
has just produced the following three-act audit of the 2025 Matai'an Creek barrier
lake event. Write a 250-word operational brief covering:

1. Three-act timeline — what the imagery proves
2. The pre-breach warning window (Jul 21 to Sep 23, 64 days) — what ARIA v5.0
   could have caught if it had been operational
3. Coverage gap — why did the W3/W7 Hualien City focus miss Guangfu?
4. Next-24-hour orders: priority clearance, shelter resupply, UAV tasking
5. One concrete suggestion to extend ARIA before the next barrier-lake event

IMPACT TABLE:
{{impact_df.to_markdown(index=False) if 'impact_df' in dir() else 'TODO'}}

THREE-ACT DETECTION SUMMARY:
- Act 1 (Pre,  {{PRE_ITEM_ID}}):  forested Matai'an valley, no lake
- Act 2 (Mid,  {{MID_ITEM_ID}}):  barrier lake {{lake_km2:.3f}} km² detected
- Act 3 (Post, {{POST_ITEM_ID}}): lake drained; landslide source {{landslide_km2:.3f}} km²;
  debris flow footprint {{debris_km2:.3f}} km² over Guangfu
"""

print(prompt)


In [ ]:
# [S14] Call your preferred LLM — THREE vendor-neutral examples
# Uncomment and fill in ONE of the three. You do NOT need all three.

# --- Example A: OpenAI SDK ---
# from openai import OpenAI
# resp = OpenAI().chat.completions.create(
#     model="gpt-4o-mini",
#     messages=[{"role": "user", "content": prompt}],
# )
# print(resp.choices[0].message.content)

# --- Example B: Google Gemini SDK ---
# import google.generativeai as genai
# genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
# model = genai.GenerativeModel("gemini-1.5-flash")
# print(model.generate_content(prompt).text)

# --- Example C: Anthropic Claude SDK ---
# import anthropic
# resp = anthropic.Anthropic().messages.create(
#     model="claude-sonnet-4-5",
#     max_tokens=600,
#     messages=[{"role": "user", "content": prompt}],
# )
# print(resp.content[0].text)


## Wrap-up Checklist / 收尾確認清單

Before leaving class, make sure you have:
離開教室前，確認你已完成：

- [ ] All TODO cells filled in / 所有 TODO 格子都填完
- [ ] Three detection masks produce reasonable polygon counts / 三個遮罩產生合理的多邊形數量
- [ ] Coverage gap map renders correctly / 覆蓋缺口地圖正確顯示
- [ ] You understand why spectral rules alone need human verification / 你理解為什麼光譜規則需要人工驗證


In [ ]:
# [S15] .env template — copy to your .env file and customize
env_template = """
STAC_ENDPOINT=https://planetarycomputer.microsoft.com/api/stac/v1
S2_COLLECTION=sentinel-2-l2a
S2_CLOUD_MAX=20
S2_BANDS=B02,B03,B04,B08,B11,B12

MATAIAN_BBOX=121.28,23.56,121.52,23.76
TARGET_EPSG=32651

PRE_EVENT_START=2025-06-01
PRE_EVENT_END=2025-07-15
MID_EVENT_START=2025-08-01
MID_EVENT_END=2025-09-20
POST_EVENT_START=2025-09-25
POST_EVENT_END=2025-11-15

AI_API_KEY=your-api-key-here
"""
print(env_template)


---

## Save Your Work / 儲存你的成果

Save this notebook with all cells executed. Your instructor may review it.
將此 notebook 連同所有執行結果一起儲存。老師可能會檢閱。
